In [8]:
import json
import sys
import os
import time
from tqdm import tqdm

from openai import OpenAI
from dotenv import load_dotenv


from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage


# Load environment variables from current directory .env and override existing ones
load_dotenv(".env", override=True)

# load api key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI(api_key=api_key)

In [14]:
with open(r'C:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\ConsensusSelection\consensus_principle_selection_th=0.42_complete_linkage.json', 'r',encoding='utf-8') as f:
    scored_principles = json.load(f)

print(scored_principles[0])


clustering_tuples = []
for item in tqdm(scored_principles):
    principle = item['summary_text']
    embedding = get_embedding(principle)
    score = item['global_msd']
    clustering_tuples.append((principle, embedding, score))


{'rank': 1, 'cluster_id': 62, 'summary_text': 'AI should be honest and truthful.', 'global_msd': 0.0004292603555339397}


100%|██████████| 276/276 [02:19<00:00,  1.99it/s]


In [11]:
# function for getting embeddings from API
def get_embedding(principle):
    num_retries = 0
    max_retries = 5
    while True:
        try:
            response = client.embeddings.create(
                input=principle,
                model="text-embedding-3-small"
            )
            return response.data[0].embedding
        except Exception as e:
            print(f"Error: {e}. Retrying in 5 seconds...")
            num_retries += 1
            if num_retries >= max_retries:
                print("Max retries reached. Exiting.")
                sys.exit(1)
            time.sleep(5)


In [63]:
# cluster 
embeddings = []
for item in clustering_tuples:
    embeddings.append(item[1])


clustering_model = AgglomerativeClustering(n_clusters=None, linkage = 'complete', metric = 'cosine', distance_threshold=.62)  # Adjust distance_threshold as needed
labels = clustering_model.fit_predict(embeddings)


In [64]:
#print number of clusters
num_clusters = len(set(labels))
print(f"Number of clusters: {num_clusters}")

#save clusters & principles to json
clusters = {}
for i, label in enumerate(labels):
    if label not in clusters:
        clusters[label] = []
    clusters[label].append((clustering_tuples[i][0], clustering_tuples[i][2]))

clusters_output = []

for i in range(len(clusters)):
    clusters_output.append({
        "cluster_id": i,
        "principles": clusters[i]
    })

with open('intermediate_principle_clusters.json', 'w') as f:
    json.dump(clusters_output, f, indent=4)

Number of clusters: 8


In [65]:
print(clusters_output[0])

# Get highest scoring principle per cluster

deduped_principles = []

for cluster in clusters_output:
    highest_scorer = min(cluster['principles'], key=lambda x: x[1])

    deduped_principles.append(highest_scorer)
    

deduped_principles.sort(key=lambda x: x[1])

print(deduped_principles)

with open('deduped_12_22_decontextual.json', 'w') as f:
    json.dump(deduped_principles, f, indent=4)

{'cluster_id': 0, 'principles': [('AI should be honest and truthful.', 0.0004292603555339397), ('AI should provide truthful and accurate information.', 0.00043424989305303115), ('AI should provide factual and accurate information.', 0.0004485807472906802), ('AI should provide responses that are accurate, informative, and thorough.', 0.00044965360549654185), ('AI should communicate in a clear and concise manner.', 0.0004518527653393377), ('AI should consistently provide factual information.', 0.0004526389920709098), ('AI should provide helpful information while avoiding responses that could cause harm or significant controversy.', 0.00045315186773976345), ('AI should consistently act with honesty and avoid any form of deception or misinformation.', 0.0004552854704974493), ('AI should provide factual, accurate, and informative responses.', 0.0004568440512599084), ('AI should always provide truthful and accurate information without deception or fabrication.', 0.00045826501712265227), ('AI